# Update transportation sector data
2026-03-12

In [121]:
import numpy as np
import pandas as pd
import sqlite3
import os

## Input data

In [122]:
data_vehicle_sales_by_size = pd.read_excel('Input data/vehicle_sales_by_size.xlsx')
data_vehicle_sales_by_powertrain = pd.read_excel('Input data/vehicle_sales_by_powertrain.xlsx')
data_survival_curve_transport = pd.read_excel('Input data/survival_curve_transport.xlsx')
data_c2a_vehicle = pd.read_excel('Input data/c2a_vehicle.xlsx')
data_activity_ceud = pd.read_excel('Input data/activity.xlsx')
data_demand_growth_rate = pd.read_excel('Input data/demand_growth_rate.xlsx')

## Self-defined functions

### Organize activity data from CEUD

In [123]:
def organize_activity_ceud(activity_ceud):

    # organize the activity data from ceud
    activity_ceud_organized = activity_ceud.melt(id_vars = ['Region', 'Mode', 'Unit'], var_name = 'period', value_name = 'demand')
    activity_ceud_organized['period'] = activity_ceud_organized['period'].astype(int)

    # map mode to commodity
    map_comm = {'Cars': 'T_D_pkm_ldv_c',
                'Light Trucks Passengers': 'T_D_pkm_ldv_t',
                'Light Trucks Freight': 'T_D_tkm_ldv_t',
                'Medium Trucks': 'T_D_tkm_mdv_t',
                'Heavy Trucks': 'T_D_tkm_hdv_t'}
    
    activity_ceud_organized['commodity'] = activity_ceud_organized['Mode'].map(map_comm)
    
    # change unit from million to billion
    activity_ceud_organized['demand'] = activity_ceud_organized['demand'] / 1000

    # change unit from billion to million
    activity_ceud_organized['units'] = activity_ceud_organized['Unit'].str.replace('M ', 'b')

    # change column name
    activity_ceud_organized = activity_ceud_organized.rename(columns = {'Region': 'region'})

    # only keep columns needed for demand table
    activity_ceud_organized = activity_ceud_organized[['region', 'commodity', 'period', 'demand', 'units']]


    return activity_ceud_organized

### Project demand for vehicle technologies

In [124]:
def project_demand_v(activity_ceud_organized, demand_growth_rate, growth_rate_type):
    """
    Project demand for vehicle technologies based on activity data and demand growth rate.
    
    Parameters:
    - activity_ceud_organized: DataFrame containing organized activity data
    - demand_growth_rate: DataFrame containing demand growth rates
    - growth_rate_type: str, type of growth rate to use ('gdp', 'population', etc.)
    
    Returns:
    - demand_update_v: DataFrame containing projected demand for vehicle technologies for each period
    """
    
    # only keep activity data for period 2023
    activity_2023 = activity_ceud_organized[activity_ceud_organized['period'] == 2023]
    activity_2023 = activity_2023.drop(columns = ['period'])

    # filter demand growth rate for the specified growth rate type
    demand_growth_rate_filtered = demand_growth_rate[['Year', growth_rate_type]]
    demand_growth_rate_filtered = demand_growth_rate_filtered.rename(columns = {'Year': 'period', growth_rate_type: 'growth_factor'})

    # merge activity data with demand growth rate
    demand_update_v = pd.merge(activity_2023, demand_growth_rate_filtered, how = 'cross')

    # calculate projected demand for each period: 2023 demand * growth factor
    demand_update_v['demand'] = demand_update_v['demand'] * demand_update_v['growth_factor']

    # only keep needed periods (original periods from 2025 to 2050)
    demand_update_v = demand_update_v[demand_update_v['period'].isin(range(2025, 2051, 5))]

    # organize format
    demand_update_v['notes'] = 'Demand projections are indexed through projected national GDP growth (2023=1), using estimates from the Current Measures Scenario of the Canada Energy Futures 2026. Initial demand is obtained from 2023 subsector activity.'
    demand_update_v['data_source'] = 'T17'
    demand_update_v['data_id'] = 'TRPHR' + demand_update_v['region'] + '001'
    demand_update_v = demand_update_v[['region', 'commodity', 'period', 'demand', 'units', 'notes', 'data_source', 'data_id']]

    return demand_update_v


### Shift time periods
new period = old period - 5
new vintage = old vintage - 5

In [125]:
def shift_time_period(data_default, range_period):

    data_shifted = data_default.copy()

    # filter data to only cover transportation sector

    # if the data contains "period" in the column:
    # shift the time period by 5 years and only keep the rows with period in the specified range
    # e.g., demand for period 2030 in default canoe data will become demand for period 2025 updated data)
    if 'period' in data_shifted.columns:
        data_shifted['period'] = data_shifted['period'] - 5
        data_shifted = data_shifted[data_shifted['period'].isin(range_period)]
    
    else:
        pass
    
    # if the data contains "vintage" in the column:
    # shift the vintage by 5 years
    # e.g., efficiency for vintage 2030 in default canoe data will efficiency for vintage 2025 in updated data
    if 'vintage' in data_shifted.columns:
        data_shifted['vintage'] = data_shifted['vintage'] - 5

    else:
        pass

    # no need to distinguish existing and future technologies in the updated data, so remove the suffix ending with "_EX" and "_N" in the technology names
    if 'tech' in data_shifted.columns:
        data_shifted['tech'] = data_shifted['tech'].str.replace(r'_EX$', '', regex = True)
        data_shifted['tech'] = data_shifted['tech'].str.replace(r'_N$', '', regex = True)

        ## drop duplicates if there are any after removing the suffix (especially for those without vintage column)
        data_shifted = data_shifted.drop_duplicates()

    else:
        pass

    return data_shifted



### Compile new data for existing capacity
Change data from stock to sales for road transportation (LDVs, MDVs, and HDV trucks)

In [126]:
def compile_data_existing_capacity(vehicle_sales_by_size, vehicle_sales_by_powertrain,
                                   range_vin):
    
    # pivot both tables to long format
    veh_sales_size = vehicle_sales_by_size.melt(id_vars = ['Region', 'Mode', 'Unit'],
                                                var_name = 'Year',
                                                value_name = 'total_sales')
    
    veh_sales_p = vehicle_sales_by_powertrain.melt(id_vars = ['Region', 'Mode', 'Powertrain'],
                                                  var_name = 'Year',
                                                  value_name = 'market_share')
    
    # join
    capa_exist = pd.merge(veh_sales_size, veh_sales_p,
                          how = 'left',
                          on = ['Region', 'Mode', 'Year'])
    
    # calculate sales by powertrain
    capa_exist['sales'] = capa_exist['total_sales'] * capa_exist['market_share']

    # drop columns
    capa_exist = capa_exist.drop(columns = ['total_sales', 'market_share'])

    # replace na with 0
    capa_exist['sales'] = capa_exist['sales'].fillna(0)

    # interpolate data for 2024-2025: assume the cloest data point (2023) remains unchanged
    ## cross join with the full year range needed
    capa_exist_temp = pd.merge(pd.DataFrame({'Year': range_vin}),
                          capa_exist[['Region', 'Mode', 'Unit', 'Powertrain']].drop_duplicates(), 
                          how = 'cross')

    ## right join the full year range needed
    exca_update_v = pd.merge(capa_exist,
                              capa_exist_temp, 
                              how = 'right',
                              on = ['Year', 'Region', 'Mode', 'Unit', 'Powertrain'])
    
    ## interpolate sales for each region, mode, powertrain
    exca_update_v = exca_update_v.sort_values(by = ['Region', 'Mode', 'Powertrain', 'Year'])
    exca_update_v['sales'] = (exca_update_v
                                 .groupby(['Region', 'Mode', 'Powertrain'], group_keys=False)['sales']
                                 .transform(lambda g: g.interpolate(method='linear', limit_direction = 'forward')))
    
    # re-categorize vintage year
    exca_update_v['vintage'] = np.nan

    for i in range(len(exca_update_v)):
        y = exca_update_v.iloc[i]['Year']

        if y == 2005:
            exca_update_v.iloc[i, exca_update_v.columns.get_loc('vintage')] = 2000
        elif 2006 <= y <= 2010:
            exca_update_v.iloc[i, exca_update_v.columns.get_loc('vintage')] = 2005
        elif 2011 <= y <= 2015:
            exca_update_v.iloc[i, exca_update_v.columns.get_loc('vintage')] = 2010
        elif 2016 <= y <= 2020:
            exca_update_v.iloc[i, exca_update_v.columns.get_loc('vintage')] = 2015
        elif 2021 <= y <= 2025:
            exca_update_v.iloc[i, exca_update_v.columns.get_loc('vintage')] = 2020
    
    # sum sales for each vintage, region, mode, powertrain
    exca_update_v = exca_update_v.groupby(['Region', 'Mode', 'Powertrain', 'vintage', 'Unit'], as_index = False).agg({'sales': 'sum'})
    
    # remove rows with 0 sales
    exca_update_v = exca_update_v[exca_update_v['sales'] > 0]

    # rename technology
    ## mode
    map_mode = {'Cars': 'T_LDV_C_',
                'Light Trucks Passengers': 'T_LDV_LTP_',
                'Light Trucks Freight': 'T_LDV_LTF_',
                'Medium Trucks': 'T_MDV_T_',
                'Heavy Trucks': 'T_HDV_T_'}
    
    exca_update_v['tech_mode'] = exca_update_v['Mode'].map(map_mode)

    ## powertrain
    map_p = {'Gasoline': 'GSL',
             'Diesel': 'DSL',
             'Battery electric': 'BEV',
             'Hybrid electric': 'GSL_HEV',
             'Plug-in hybrid electric': 'GSL_PHEV35'}
    
    exca_update_v['tech_p'] = exca_update_v['Powertrain'].map(map_p)

    ### special case for BEV in LDV_C (BEV150) and LDV_LTP (BEV300)
    exca_update_v.loc[(exca_update_v['Mode'] == 'Cars') & (exca_update_v['Powertrain'] == 'Battery electric'), 'tech_p'] = 'BEV150'
    exca_update_v.loc[(exca_update_v['Mode'] == 'Light Trucks Passengers') & (exca_update_v['Powertrain'] == 'Battery electric'), 'tech_p'] = 'BEV300'

    ## combine
    exca_update_v['tech'] = exca_update_v['tech_mode'] + exca_update_v['tech_p']

    # rename columns
    exca_update_v = exca_update_v.rename(columns = {'Region': 'region',
                                              'sales': 'capacity',
                                              'Unit': 'units'})
    
    # select columns
    exca_update_v = exca_update_v[['region', 'tech', 'vintage', 'capacity', 'units']]

    # get the combinations of region, tech, vintage in the existing capacity data
    tech_hist_update_v = exca_update_v[['region', 'tech', 'vintage']].drop_duplicates()

    # set data_id for exca_update_v
    exca_update_v['data_id'] = 'TRPHR' + exca_update_v['region'] + '001'
    
    return exca_update_v, tech_hist_update_v

### Compile efficiency and variable costs data for new combinations of existing capacity

In [127]:
def compile_data_eff_cova(tech_hist_shifted_v, tech_hist_update_v,
                          eff_shifted_v, cova_shifted_v,
                          range_period):

    # get the combinations of region, vintage, tech that need to be dropped from the default table
    tech_hist_drop_v = tech_hist_update_v.merge(tech_hist_shifted_v, how = 'right', on = ['region', 'tech', 'vintage'], indicator = True)
    tech_hist_drop_v = tech_hist_drop_v[tech_hist_drop_v['_merge'] == 'right_only']

    # get the new combinations of all region, vintage, tech (include both historical and new)
    tech_trans_update_v = pd.concat([tech_hist_update_v,
                                   eff_shifted_v[['region', 'tech', 'vintage']]], ignore_index=True).drop_duplicates()
    
    ## drop combinations in "tech_hist_drop_v"
    tech_trans_update_v = tech_trans_update_v.merge(tech_hist_drop_v[['region', 'tech', 'vintage']],
                                                how = 'left',
                                                on = ['region', 'tech', 'vintage'], indicator = True)
    
    tech_trans_update_v = tech_trans_update_v[tech_trans_update_v['_merge'] == 'left_only'].drop(columns=['_merge'])

    # add the efficiency for new combinations
    ## left join with the default efficiency table
    eff_update_v = pd.merge(tech_trans_update_v,
                   eff_shifted_v[['tech', 'input_comm', 'output_comm']].drop_duplicates(),
                   how = 'left',
                   on = ['tech'])
    
    eff_update_v = pd.merge(eff_update_v,
                            eff_shifted_v,
                            how = 'left',
                            on = ['region', 'tech', 'vintage', 'input_comm', 'output_comm'])
    
    ## extrapolate efficiency for combinations without values by using the value of the closest vintage with value
    eff_update_v = eff_update_v.sort_values(by = ['region', 'tech', 'vintage']).reset_index(drop = True)
    eff_update_v['efficiency'] = eff_update_v.groupby(['region', 'tech'])['efficiency'].ffill().bfill()
    eff_update_v['notes'] = eff_update_v.groupby(['region', 'tech'])['notes'].ffill().bfill()

    # add variable costs for new combinations
    ## cross join the full transportation technology list with all study periods
    cova_update_v = pd.merge(pd.DataFrame({'period': range_period}),
                             tech_trans_update_v,
                             how = 'cross')
    
    ## left join with the default variable cost table
    cova_update_v = pd.merge(cova_update_v,
                           cova_shifted_v,
                           how = 'left',
                           on = ['region', 'tech', 'vintage', 'period'])
    
    ## for combinations with some values for some periods but not all, extrapolate variable cost for periods without values by using the value of the closest period with value
    cova_update_v = cova_update_v.sort_values(by = ['region', 'tech', 'vintage', 'period']).reset_index(drop = True)
    cova_update_v['cost'] = (cova_update_v
                           .groupby(['region', 'tech', 'vintage'], group_keys=False)['cost']
                           .transform(lambda g: g.interpolate(method='linear', limit_direction = 'both')))
    
    ### add notes
    mask = cova_update_v['notes'].isna() & cova_update_v['cost'].notna()
    cova_update_v.loc[mask, 'notes'] = ("Added to adjust variable costs based on survival curve")

    ## for combinations without any value for all periods, extrapolate variable cost for vintages without values by using the value of the closest vintage with value in the same period
    cova_update_v = cova_update_v.sort_values(by = ['region', 'tech', 'period', 'vintage']).reset_index(drop = True)

    cova_update_v['cost'] = (cova_update_v
                           .groupby(['region', 'tech', 'period'], group_keys=False)['cost']
                           .transform(lambda g: g.interpolate(method='linear', limit_direction = 'both')))
    
    ### add notes
    cova_update_v['notes'] = cova_update_v['notes'].fillna('Assumed using the value of the closest vintage with value in the same period')

    ### extrapolate units
    cova_update_v['units'] = cova_update_v.groupby(['region', 'tech', 'vintage'])['units'].ffill().bfill()

    # set data_id
    eff_update_v['data_id'] = 'TRPHR' + eff_update_v['region'] + '001'
    cova_update_v['data_id'] = 'TRPHR' + cova_update_v['region'] + '001'

    return tech_trans_update_v, tech_hist_drop_v, eff_update_v, cova_update_v


### Compile survival curves and related data

In [ ]:
def compile_data_survival_lifetime_transport(survival_curve_transport, tech_trans_update_v, cova_update_v,
                                             fixed_lifetime, lifetime_default_v):

    transport_survival_lt = survival_curve_transport.copy()

    # if fixed_lifetime is selected, use the default lifetime to generate survival curves
    if fixed_lifetime is True:
        
        ## get the maximum lifetime from the default lifetime table
        survival_update_v = lifetime_default_v[['region', 'tech', 'lifetime', 'notes']].drop_duplicates().copy()

        ## turn lifetime into integer
        survival_update_v['lifetime'] = survival_update_v['lifetime'].astype(int)

        ## cross join with age (from 0 to the maximum lifetime of each region and tech)
        survival_update_v['Age'] = survival_update_v['lifetime'].apply(lambda x: range(x + 1))
        survival_update_v = survival_update_v.explode('Age').reset_index(drop=True)

        ## calculate fraction: if age < lifetime, fraction = 1; if age >= lifetime, fraction = 0
        survival_update_v['fraction'] = (survival_update_v['Age'] < survival_update_v['lifetime']).astype(int)

        ## set notes
        survival_update_v['notes'] = 'Assumed using fixed lifetime to generate survival curves'

        ## join with the transportation technology
        survival_update_v = pd.merge(tech_trans_update_v, survival_update_v, on = ['region', 'tech'], how = 'left')

        ## use default lifetime as lifetime
        lifetime_update_v = lifetime_default_v.copy()
    
    else:
        # organize survival rates
        ## melt survival rates to long
        transport_survival_lt = transport_survival_lt.melt(id_vars = ['Age'],
                                            var_name = 'Mode',
                                            value_name = 'fraction')
        
        transport_survival_lt['notes'] = 'From MOVES5'
        
        # get the lifetime
        transport_lt = transport_survival_lt.groupby('Mode', as_index = False).agg({'Age': 'max'})
        transport_lt = transport_lt.rename(columns = {'Age': 'lifetime'})
        transport_lt['notes'] = '(y) age when vehicles have 0% survival rate'

        # join with the transportation technology
        ## get modes for transportation technology
        rules = [("T_LDV_C", "Cars"),
                ("T_LDV_LTP", "Light Trucks Passenger"),
                ("T_LDV_LTF", "Light Trucks Freight"),
                ("T_MDV_T", "Medium Trucks"),
                ("T_HDV_T", "Heavy Trucks"),]
        
        # built vectorized condictions
        conditions = [
            tech_trans_update_v['tech'].str.contains(sub, case=False, na=False)
            for sub, _ in rules
        ]

        choices = [label for _, label in rules]
                
        tech_trans_update_v['Mode'] = np.select(conditions, choices, default=tech_trans_update_v['tech'])

        ## left join with the survival curve table
        survival_update_v = pd.merge(tech_trans_update_v,
                                    transport_survival_lt,
                                    how = 'left',
                                    on = ['Mode'])
        
        ## left join with lifetime table
        lifetime_update_v = pd.merge(tech_trans_update_v.drop(columns = ['vintage']).drop_duplicates(),
                                 transport_lt,
                                 how = 'left',
                                 on = ['Mode'])
    
        ### drop columns
        lifetime_update_v = lifetime_update_v[['region', 'tech', 'lifetime', 'notes']]
    
    ### calculate period
    survival_update_v['period'] = survival_update_v['vintage'] + survival_update_v['Age']

    ### only keep two digits for fraction
    survival_update_v['fraction'] = survival_update_v['fraction'].round(2)

    ### drop columns
    survival_update_v = survival_update_v[['region', 'tech', 'vintage', 'period', 'fraction', 'notes']]

    # edit variable costs: remove rows where age < 0 or age > lifetime
    ## left join with lifetime table to get lifetime for each technology
    cova_update_v = cova_update_v.merge(lifetime_update_v[['tech', 'lifetime']].drop_duplicates(),
                                        on = 'tech',
                                        how = 'left')
    
    ## get age for each row
    cova_update_v['age'] = cova_update_v['period'] - cova_update_v['vintage']

    ## remove rows where age < 0 or age >= lifetime
    cova_update_v = cova_update_v[(cova_update_v['age'] >= 0) & (cova_update_v['age'] < cova_update_v['lifetime'])]

    ## drop columns
    cova_update_v = cova_update_v.drop(columns = ['age', 'lifetime'])

    # set data_id
    survival_update_v['data_id'] = 'TRPHR' + survival_update_v['region'] + '001'
    lifetime_update_v['data_id'] = 'TRPHR' + lifetime_update_v['region'] + '001'

    
    
    return survival_update_v, lifetime_update_v, cova_update_v
    

### Compile capacity to activity data

In [129]:
def compile_data_c2a(c2a_vehicle, c2a_shifted_o):

    # get regions from c2a_shifted_o
    regions_c2a = c2a_shifted_o['region'].unique()

    # organize c2a for vehicles
    c2a_update_v = c2a_vehicle.copy()

    ## left join regions
    c2a_update_v = pd.merge(c2a_update_v,
                            pd.DataFrame({'region': regions_c2a}),
                            how = 'cross')
    
    ## arrange columns
    c2a_update_v = c2a_update_v[['region', 'tech', 'c2a', 'notes']]

    ## add data_id
    c2a_update_v['data_id'] = 'TRPHR' + c2a_update_v['region'] + '001'

    ## concatenate with c2a for non-vehicles
    c2a_update = pd.concat([c2a_update_v, c2a_shifted_o], ignore_index = True)

    return c2a_update, c2a_update_v

### Compile annual capacity factor

#### For existing vehicles: simulate by region

In [130]:
def compile_data_lacf_v_ex(exca_update_v, survival_update_v,
                           c2a_update_v, eff_update_v, activity_ceud_organized):

    # calculate capacity existing active in the year of 2023
    ## filter survival curves to only keep those in the year of 2023 (survival curve is annual instead of 5-year period)
    surv_EX_2023 = survival_update_v[survival_update_v['period'] == 2023]

    ## merge with capa_exist_filter to calculate stock in 2023
    stock_2023 = exca_update_v.merge(surv_EX_2023, on = ['region', 'tech', 'vintage'])
    stock_2023['stock_2023: {k units}'] = stock_2023['capacity'] * stock_2023['fraction']
    stock_2023 = stock_2023[['region', 'tech', 'vintage', 'stock_2023: {k units}']]
    
    # calculate maximum supply in the year of 2023: stock * c2a
    ## merge stock and c2a
    supply_max_2023 = stock_2023.merge(c2a_update_v[['region', 'tech', 'c2a']].drop_duplicates(),
                                         how = 'left',
                                         on = ['region', 'tech'])
    
    ## calculate maximum supply by region, tech, vintage
    supply_max_2023['supply_max_2023: {bpkm or btkm}'] = supply_max_2023['stock_2023: {k units}'] * supply_max_2023['c2a']

    ## left join with efficiency
    supply_max_2023 = supply_max_2023.merge(eff_update_v[['region', 'tech', 'vintage', 'output_comm']].drop_duplicates(),
                                              how = 'left',
                                              on = ['region', 'tech', 'vintage'])
    
    ## calculate maximum supply by region and output commodity
    supply_max_2023 = supply_max_2023.groupby(['region', 'output_comm'], as_index = False).agg({'supply_max_2023: {bpkm or btkm}': 'sum'})
    
    # calculate LACF
    ## get actual demand during 2023
    demand_p2023 = activity_ceud_organized[activity_ceud_organized['period'] == 2023]
    demand_p2023 = demand_p2023.rename(columns = {'demand': 'demand_p2023: {bpkm or btkm}',
                                                  'commodity': 'output_comm'})
    
    ## left join with actual demand in 2023
    lacf_update_v_ex = demand_p2023.merge(supply_max_2023,
                                          how = 'left',
                                          on = ['region', 'output_comm'])
    
    ## calculate LACF = actual demand in period "2023" / maximum supply during period "2023"
    lacf_update_v_ex['lacf_update_v'] = lacf_update_v_ex['demand_p2023: {bpkm or btkm}'] / lacf_update_v_ex['supply_max_2023: {bpkm or btkm}']

    ## only keep rows with 'lacf_update_v' is not na
    lacf_update_v_ex = lacf_update_v_ex[lacf_update_v_ex['lacf_update_v'].notna()]

    ## only keep columns needed
    lacf_update_v_ex = lacf_update_v_ex[['region', 'output_comm', 'lacf_update_v', 'demand_p2023: {bpkm or btkm}', 'supply_max_2023: {bpkm or btkm}']]

    ## calculate lower and upper bound (+/- 2.5%)
    lacf_update_v_ex['LACF_le_update'] = lacf_update_v_ex['lacf_update_v'] * (1 + 0.025)
    lacf_update_v_ex['LACF_ge_update'] = lacf_update_v_ex['lacf_update_v'] * (1 - 0.025)

    # left join with vintage for efficiency update
    eff_update_v_ex = eff_update_v[eff_update_v['vintage'] < 2025]

    lacf_update_v_ex = pd.merge(lacf_update_v_ex,
                                eff_update_v_ex[['region', 'output_comm', 'vintage', 'tech']].drop_duplicates(),
                                how = 'left',
                                on = ['region', 'output_comm'])
    
    # pivot to long format
    lacf_update_v_ex = lacf_update_v_ex.melt(id_vars = ['region', 'tech', 'vintage', 'output_comm'],
                                   value_vars = ['LACF_le_update', 'LACF_ge_update'],
                                   var_name = 'operator',
                                   value_name = 'factor')

    # rename operator values
    lacf_update_v_ex['operator'] = lacf_update_v_ex['operator'].map({'LACF_le_update': 'le',
                                                                     'LACF_ge_update': 'ge'})
    
    return lacf_update_v_ex

#### For new vehicles: simulate national

In [131]:
def compile_data_lacf_v_n(exca_update_v, survival_update_v,
                          c2a_update_v, eff_update_v, activity_ceud_organized):

    # calculate capacity existing active in the year of 2023
    ## filter survival curves to only keep those in "2023" (survival curve data is annual instead of 5-year)
    surv_EX_2023 = survival_update_v[survival_update_v['period'] == 2023]

    ## merge with capa_exist_filter to calculate stock in 2023
    stock_2023 = exca_update_v.merge(surv_EX_2023, on = ['region', 'tech', 'vintage'])
    stock_2023['stock_2023: {k units}'] = stock_2023['capacity'] * stock_2023['fraction']
    stock_2023 = stock_2023[['region', 'tech', 'vintage', 'stock_2023: {k units}']]

    # calculate maximum supply in the year of 2023: stock * c2a
    ## merge stock and c2a
    supply_max_2023 = stock_2023.merge(c2a_update_v[['region', 'tech', 'c2a']].drop_duplicates(),
                                        how = 'left',
                                        on = ['region', 'tech'])
    
    ## calculate maximum supply by region, tech, vintage
    supply_max_2023['supply_max_2023: {bpkm or btkm}'] = supply_max_2023['stock_2023: {k units}'] * supply_max_2023['c2a']

    ## left join with efficiency
    supply_max_2023 = supply_max_2023.merge(eff_update_v[['region', 'tech', 'vintage', 'output_comm']].drop_duplicates(),
                                            how = 'left',
                                            on = ['region', 'tech', 'vintage'])
    
    ## calculate maximum supply by output commodity
    supply_max_2023 = supply_max_2023.groupby(['output_comm'], as_index = False).agg({'supply_max_2023: {bpkm or btkm}': 'sum'})
    
    # calculate LACF
    ## get actual demand during period "2023"
    demand_p2023 = activity_ceud_organized[activity_ceud_organized['period'] == 2023]
    demand_p2023 = demand_p2023.rename(columns = {'demand': 'demand_p2023: {bpkm or btkm}',
                                                  'commodity': 'output_comm'})

    ### sum national demand for each output commodity
    demand_p2023 = demand_p2023.groupby(['output_comm'], as_index = False).agg({'demand_p2023: {bpkm or btkm}': 'sum'})
    
    ## left join with actual demand in 2023
    lacf_update_v_n = demand_p2023.merge(supply_max_2023,
                                          how = 'left',
                                          on = ['output_comm'])
    
    ## calculate LACF = actual demand in period "2023" / maximum supply during period "2023"
    lacf_update_v_n['lacf_update_v'] = lacf_update_v_n['demand_p2023: {bpkm or btkm}'] / lacf_update_v_n['supply_max_2023: {bpkm or btkm}']

    ## only keep rows with 'lacf_update_v' is not na
    lacf_update_v_n = lacf_update_v_n[lacf_update_v_n['lacf_update_v'].notna()]

    ## only keep columns needed
    lacf_update_v_n = lacf_update_v_n[['output_comm', 'lacf_update_v', 'demand_p2023: {bpkm or btkm}', 'supply_max_2023: {bpkm or btkm}']]

    ## calculate lower and upper bound (+/- 2.5%)
    lacf_update_v_n['LACF_le_update'] = lacf_update_v_n['lacf_update_v'] * (1 + 0.025)
    lacf_update_v_n['LACF_ge_update'] = lacf_update_v_n['lacf_update_v'] * (1 - 0.025)

    # left join with vintage for efficiency update
    eff_update_v_n = eff_update_v[eff_update_v['vintage'] >= 2025]

    lacf_update_v_n = pd.merge(lacf_update_v_n,
                               eff_update_v_n[['region', 'output_comm', 'vintage', 'tech']].drop_duplicates(),
                               how = 'left',
                               on = ['output_comm'])
    
    # pivot to long format
    lacf_update_v_n = lacf_update_v_n.melt(id_vars = ['region', 'tech', 'vintage', 'output_comm'],
                                           value_vars = ['LACF_le_update', 'LACF_ge_update'],
                                           var_name = 'operator',
                                           value_name = 'factor')

    # rename operator values
    lacf_update_v_n['operator'] = lacf_update_v_n['operator'].map({'LACF_le_update': 'le',
                                                                   'LACF_ge_update': 'ge'})
    

    return lacf_update_v_n


#### Combine with non-transportation technologies

In [132]:
def compile_lacf_transport(lacf_update_v_ex, lacf_update_v_n, lacf_shifted_o, eff_shifted_o):

    # get data_id for lacf for vehicle technologies
    lacf_update_v = pd.concat([lacf_update_v_ex, lacf_update_v_n], ignore_index = True)
    lacf_update_v['data_id'] = 'TRPHR' + lacf_update_v['region'] + '001'

    # organize data for lacf for non-vehicle technologies
    lacf_update_o = lacf_shifted_o.copy()

    ## remove column "period"
    lacf_update_o = lacf_update_o.drop(columns = ['period'])
    lacf_update_o = lacf_update_o.drop_duplicates()

    ## left join with vintage from efficiency table
    lacf_update_o = pd.merge(lacf_update_o,
                             eff_shifted_o[['region', 'vintage', 'tech']].drop_duplicates(),
                             how = 'left',
                             on = ['region', 'tech'])

    ## move "vintage" column next to "tech" column
    cols = lacf_update_o.columns.tolist()
    cols.insert(cols.index('tech') + 1, cols.pop(cols.index('vintage')))
    lacf_update_o = lacf_update_o[cols]
    
    # concatenate lacf for vehicle and non-vehicle technologies
    lacf_update = pd.concat([lacf_update_o, lacf_update_v], ignore_index = True)

    return lacf_update

### Remove data for existing technologies that do not have existing capacities

In [133]:
def remove_data_noexca(data_update, range_period, exca_update):

    # separate data from existing technologies and new technologies
    data_update_ex = data_update[data_update['vintage'] < min(range_period)]
    data_update_n = data_update[data_update['vintage'] >= min(range_period)]

    # remove data for new technologies that are not in the existing capacity data
    data_update_ex_clean = data_update_ex.merge(exca_update[['region', 'tech', 'vintage']].drop_duplicates(),
                                                how = 'left',
                                                on = ['region', 'tech', 'vintage'], indicator = True)
    
    data_update_ex_clean = data_update_ex_clean[data_update_ex_clean['_merge'] == 'both'].drop(columns=['_merge'])

    # concatenate data for existing technologies and new technologies
    data_update_clean = pd.concat([data_update_ex_clean, data_update_n], ignore_index=True)

    return data_update_clean

## Main: compile data

In [134]:
def main_compile_data_transport(canoe_default_all,
                                vehicle_sales_by_size,
                                vehicle_sales_by_powertrain,
                                range_vin, range_period,
                                survival_curve_transport,
                                c2a_vehicle,
                                activity_ceud,
                                demand_growth_rate, growth_rate_type,
                                fixed_lifetime):
    
    
    # extract data from canoe default
    connection = sqlite3.connect(canoe_default_all)

    ## get the tables in default canoe data which have data related to the transportation sector
    tables = ['Demand', 'Efficiency', 'CostInvest', 'CostVariable', 'CostFixed', 
              'ExistingCapacity', 'TimePeriod', 
              'CapacityFactorTech', 'LimitAnnualCapacityFactor', 'LifetimeTech', 'LimitTechInputSplitAnnual',
              'CapacityToActivity', 'Commodity', 'TechGroupMember', 'Technology', 'EmissionActivity']
    demand_default, eff_default, coin_default, cova_default, cofi_default, \
    exca_default, tipe_default, cft_default, lacf_default, lifetime_default, tein_default, \
    c2a_default, comm_default, tgm_default, tech_default, emac_default = \
        [pd.read_sql_query(f"SELECT * FROM {table}", connection) for table in tables]

    connection.close()

    # only keep the rows related to transportation sector for the parameters that contain data for multiple sectors
    ## for demand and commodity
    demand_default = demand_default[demand_default['commodity'].str.contains(r'^T_', na=False)]
    comm_default = comm_default[comm_default['name'].str.contains(r'^T_', na=False)]     
    
    ## for other tables (transprotation: anything start with "T_" in "tech")
    ### get the technology list for transportation sector based on the technology table
    tech_list = tech_default[tech_default['sector'] == 'transportation']['tech'].unique().tolist()

    eff_default, coin_default, cova_default, cofi_default, exca_default, cft_default, lacf_default, lifetime_default, tein_default, c2a_default, tgm_default, tech_default, emac_default = [
        df[df['tech'].isin(tech_list)]
        for df in [eff_default, coin_default, cova_default, cofi_default, exca_default, cft_default, lacf_default, lifetime_default, tein_default, c2a_default, tgm_default, tech_default, emac_default]
    ]

    ## organize activity data from ceud
    activity_ceud_organized = organize_activity_ceud(activity_ceud)

    # for demand: replace vehicle demand
    ## separate demand for vehicle technologies and non-vehicle technologies
    demand_default_o = demand_default[~demand_default['commodity'].isin(['T_D_pkm_ldv_c', 'T_D_pkm_ldv_t', 'T_D_tkm_ldv_t', 'T_D_tkm_mdv_t', 'T_D_tkm_hdv_t'])]

    ## get demand for vehicle technologies from activity data
    demand_update_v = project_demand_v(activity_ceud_organized, demand_growth_rate, growth_rate_type)

    ## combine demand for vehicle technologies and non-vehicle technologies
    demand_update = pd.concat([demand_default_o, demand_update_v], ignore_index=True)
    
    # shift time priod
    ## shift time period for all techno-economic parameters
    demand_update_shifted, eff_shifted, coin_shifted, cova_shifted, cofi_shifted, \
    exca_shifted, cft_shifted, lacf_shifted, tein_shifted, emac_shifted = [
        shift_time_period(data_default=df, range_period=range_period)
        for df in [demand_update, eff_default, coin_default, cova_default, cofi_default, 
                   exca_default, cft_default, lacf_default, tein_default, emac_default]
    ]

    ## for investment costs, remove vintages before the start of modeling period
    coin_shifted = coin_shifted[coin_shifted['vintage'] >= min(range_period)]
    
    ## remove period "2055" from time period table since it's not needed for the updated data
    tipe_update = tipe_default[tipe_default['period'] != 2055]

    # for c2a, tech, tech group, lifetime: no need to shift time period but need to change technology names to remove the suffix for existing and new technologies
    c2a_update, tech_update, tgm_update, lifetime_default = [
        df.assign(tech=df['tech'].str.replace(r'_EX$', '', regex=True).str.replace(r'_N$', '', regex=True)).drop_duplicates()
        for df in [c2a_default, tech_default, tgm_default, lifetime_default]
    ]

    # for LDVs (cars, LTP, LTF), MDVs, and HDV trucks (all named as 'v' to refer to vehicles)
    ## filter the default efficiency and variable cost tables to only keep the rows for transportation technologies (LDVs, MDVs, HDV trucks) for later use in compiling efficiency and variable cost data
    tech_list_v = ['T_LDV_C', 'T_LDV_LTP', 'T_LDV_LTF', 'T_MDV_T', 'T_HDV_T']
    
    eff_shifted_v = eff_shifted[eff_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    cova_shifted_v = cova_shifted[cova_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    exca_shifted_v = exca_shifted[exca_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    lifetime_default_v = lifetime_default[lifetime_default['tech'].str.contains('|'.join(tech_list_v), na = False)]

    ### separate data for non vehicles in transportation (named as 'o' to refer to others)
    eff_shifted_o = eff_shifted[~eff_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    cova_shifted_o = cova_shifted[~cova_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    exca_shifted_o = exca_shifted[~exca_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    c2a_shifted_o = c2a_update[~c2a_update['tech'].str.contains('|'.join(tech_list_v), na = False)]
    lacf_shifted_o = lacf_shifted[~lacf_shifted['tech'].str.contains('|'.join(tech_list_v), na = False)]
    lifetime_default_o = lifetime_default[~lifetime_default['tech'].str.contains('|'.join(tech_list_v), na = False)]

    ## get the default historical technology combinations (region, vintage, tech) based on the efficiency table
    tech_hist_shifted_v = exca_shifted_v[['region', 'vintage', 'tech']].drop_duplicates()

    ## compile new existing capacity data
    exca_update_v, tech_hist_update_v = compile_data_existing_capacity(vehicle_sales_by_size,
                                                                       vehicle_sales_by_powertrain,
                                                                       range_vin)
    
    ## compile efficiency and variable costs
    tech_trans_update_v, tech_hist_drop_v, eff_update_v, cova_update_v = compile_data_eff_cova(tech_hist_shifted_v,
                                                                             tech_hist_update_v,
                                                                             eff_shifted_v,
                                                                             cova_shifted_v,
                                                                             range_period)
    
    ## compile survival curves and related data
    survival_update_v, lifetime_update_v, cova_update_v = compile_data_survival_lifetime_transport(survival_curve_transport,
                                                                                                   tech_trans_update_v,
                                                                                                   cova_update_v,
                                                                                                   fixed_lifetime = fixed_lifetime,
                                                                                                   lifetime_default_v = lifetime_default_v)
    
    ## compile capacity to activity
    c2a_update, c2a_update_v = compile_data_c2a(c2a_vehicle, c2a_shifted_o)

    ## compile LACF
    ### existing vehicles: use simulated values by region
    lacf_update_v_ex = compile_data_lacf_v_ex(exca_update_v, survival_update_v, c2a_update_v, eff_update_v, activity_ceud_organized)
    
    ### new vehicles: use national simulated values
    lacf_update_v_n = compile_data_lacf_v_n(exca_update_v, survival_update_v, c2a_update_v, eff_update_v, activity_ceud_organized)

    ### combine vehicle and non-vehicle technologies
    lacf_update = compile_lacf_transport(lacf_update_v_ex,
                                         lacf_update_v_n,
                                         lacf_shifted_o,
                                         eff_shifted_o)
    
    # replace data in shifted tables with the updated data for transportation technologies
    eff_update = pd.concat([eff_shifted_o, eff_update_v], ignore_index=True)
    cova_update = pd.concat([cova_shifted_o, cova_update_v], ignore_index=True)
    exca_update = pd.concat([exca_shifted_o, exca_update_v], ignore_index=True)
    lifetime_update = pd.concat([lifetime_default_o, lifetime_update_v], ignore_index=True)

    # also remove data for existing technologies that are not in the updated data
    eff_update_clean = remove_data_noexca(data_update = eff_update, range_period = range_period, exca_update = exca_update)
    cova_update_clean = remove_data_noexca(data_update = cova_update, range_period = range_period, exca_update = exca_update)
    coin_shifted_clean = remove_data_noexca(data_update = coin_shifted, range_period = range_period, exca_update = exca_update)
    cofi_shifted_clean = remove_data_noexca(data_update = cofi_shifted, range_period = range_period, exca_update = exca_update)
    survival_update_clean_v = remove_data_noexca(data_update = survival_update_v, range_period = range_period, exca_update = exca_update)
    lacf_update_clean = remove_data_noexca(data_update = lacf_update, range_period = range_period, exca_update = exca_update)

    # write csv
    comm_default.to_csv('Output data/Commodity_transport.csv', index=False)

    tech_update.to_csv('Output data/Technology_transport.csv', index=False)
    tgm_update.to_csv('Output data/TechGroupMember_transport.csv', index=False)

    demand_update_shifted.to_csv('Output data/Demand_transport.csv', index=False)
    cft_shifted.to_csv('Output data/CapacityFactorTech_transport.csv', index=False)
    tein_shifted.to_csv('Output data/LimitTechInputSplitAnnual_transport.csv', index=False)
    emac_shifted.to_csv('Output data/EmissionActivity_transport.csv', index=False)

    #tipe_update.to_csv('Output data/TimePeriod.csv', index=False)

    eff_update_clean.to_csv('Output data/Efficiency_transport.csv', index=False)
    cova_update_clean.to_csv('Output data/CostVariable_transport.csv', index=False)
    coin_shifted_clean.to_csv('Output data/CostInvest_transport.csv', index=False)
    cofi_shifted_clean.to_csv('Output data/CostFixed_transport.csv', index=False)

    exca_update.to_csv('Output data/ExistingCapacity_transport.csv', index=False)
    c2a_update.to_csv('Output data/CapacityToActivity_transport.csv', index=False)
    lacf_update_clean.to_csv('Output data/LimitAnnualCapacityFactor_transport.csv', index=False)

    survival_update_clean_v.to_csv('Output data/LifetimeSurvivalCurve_vehicle.csv', index=False)
    lifetime_update.to_csv('Output data/LifetimeTech_transport.csv', index=False)

    #tech_hist_drop_v.to_csv('Output data/TechHistDrop_vehicle.csv', index=False)

    # compile above data into a dictionary using the table names as keys, which will be used to update the default canoe database for transportation sector
    data_update = {
        'Commodity': comm_default,
        'Technology': tech_update,
        'TechGroupMember': tgm_update,

        'Demand': demand_update_shifted,
        'CapacityFactorTech': cft_shifted,
        'LimitTechInputSplitAnnual': tein_shifted,
        'EmissionActivity': emac_shifted,

        'TimePeriod': tipe_update,

        'Efficiency': eff_update_clean,
        'CostVariable': cova_update_clean,

        'CostInvest': coin_shifted_clean,
        'CostFixed': cofi_shifted_clean,

        'ExistingCapacity': exca_update,
        'CapacityToActivity': c2a_update,
        'LimitAnnualCapacityFactor': lacf_update_clean,

        'LifetimeSurvivalCurve': survival_update_clean_v,
        'LifetimeTech': lifetime_update,
    }

    #return data_update


## Run: compile data

In [ ]:
main_compile_data_transport(canoe_default_all = 'Input data/canoe_ALL_cm.sqlite',
                            vehicle_sales_by_size = data_vehicle_sales_by_size,
                            vehicle_sales_by_powertrain = data_vehicle_sales_by_powertrain,
                            range_vin = np.arange(2005, 2026, 1),
                            range_period = range(2025, 2046, 5),
                            survival_curve_transport = data_survival_curve_transport,
                            c2a_vehicle = data_c2a_vehicle,
                            activity_ceud = data_activity_ceud,
                            demand_growth_rate = data_demand_growth_rate,
                            growth_rate_type = 'gdp',
                            fixed_lifetime = False)

  region         tech  lifetime  \
0     AB  T_HDV_T_BEV        19   
1     AB  T_HDV_T_BEV        19   
2     AB  T_HDV_T_BEV        19   
3     AB  T_HDV_T_BEV        19   
4     AB  T_HDV_T_BEV        19   

                                               notes Age  fraction  
0  Assumed using fixed lifetime to generate survi...   0         1  
1  Assumed using fixed lifetime to generate survi...   1         1  
2  Assumed using fixed lifetime to generate survi...   2         1  
3  Assumed using fixed lifetime to generate survi...   3         1  
4  Assumed using fixed lifetime to generate survi...   4         1  
  region            tech  vintage
0     AB  T_LDV_C_BEV150   2010.0
1     AB  T_LDV_C_BEV150   2015.0
2     AB  T_LDV_C_BEV150   2020.0
3     AB     T_LDV_C_DSL   2005.0
4     AB     T_LDV_C_DSL   2010.0


## Check tables whose time needs to be shifted

In [136]:
# print all table names in an sqlite with a specific column
def print_tables_with_column(sqlite_file, column_name):
    connection = sqlite3.connect(sqlite_file)
    cursor = connection.cursor()

    # get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    tables_with_period = []

    # only print table names where the table is not empty and contains the specific column name
    for table in tables:
        table_name = table[0]
        cursor.execute(f"PRAGMA table_info('{table_name}');")
        columns = cursor.fetchall()
        column_names = [column[1] for column in columns]

        if column_name in column_names and len(cursor.execute(f"SELECT * FROM '{table_name}'").fetchall()) > 0:
            tables_with_period.append(table_name)

    connection.close()
    
    return tables_with_period

In [137]:
print_tables_with_column(sqlite_file = 'Input data/canoe_ALL_cm_16d.sqlite', column_name = 'period')

['CapacityCredit',
 'CapacityFactorProcess',
 'CapacityFactorTech',
 'CostFixed',
 'CostVariable',
 'Demand',
 'DemandSpecificDistribution',
 'LimitActivity',
 'LimitAnnualCapacityFactor',
 'LimitCapacity',
 'LimitNewCapacityShare',
 'LimitSeasonalCapacityFactor',
 'LimitTechInputSplitAnnual',
 'ReserveCapacityDerate',
 'TimeSegmentFraction',
 'TimePeriod',
 'TimeSeason',
 'TimeSeasonSequential']

In [138]:
#print_tables_with_column(sqlite_file = 'Input data/canoe_ALL_cm_16d.sqlite', column_name = 'vintage')